# UrgeNurse — Benchmark de modelos OCR (imagen/PDF → texto) en CPU

Este notebook usa `ocr.py` como **paquete de funciones** para:

- **descargar/cargar** los motores OCR del catálogo (Tesseract, PaddleOCR,
  RapidOCR, EasyOCR, docTR, Docling) — CPU, ligeros, ≤ 2 GB RAM,
- **correr** cada motor sobre los 20 documentos clínicos sintéticos
  (urgencias, laboratorio, radiología, diagnósticos, órdenes de medicación),
  escribiendo una predicción por documento en `predictions/<modelo>/{n}.txt`,
- **calcular y analizar las métricas** estándar de OCR: CER, WER, precisión/recall/F1
  de palabra, WER de terminología clínica, field recall, latencia (ms/página),
  throughput (páginas/s) y RAM peak,
- **comparar con gráficas** y ejecutar la **etapa de selección del modelo**.

> **Prerrequisito:** el dataset (imágenes + ground-truth) debe existir ya en
> `assets/images/` y `assets/references/`. Se genera **una sola vez** con el
> comando aparte:
> ```bash
> python prepare_dataset.py
> ```
> que maqueta 20 documentos con **ReportLab** (PDF), los rasteriza a PNG y deja la
> ground-truth verbatim. A diferencia del ASR, la ground-truth es **exacta** (es el
> texto que se imprimió), así que no hay sesgo de transcripción.

Requiere `pandas`, `matplotlib`, `numpy`. Los backends OCR (`pytesseract`,
`paddleocr`, `rapidocr-onnxruntime`, `easyocr`, `python-doctr`, `docling`) son
opcionales: si alguno falta, su modelo se omite solo.

In [ ]:
# 1) Configuración por variables de entorno (se leen al importar ocr)
import os
from pathlib import Path

MODELS_DIR = (Path.cwd() / ".." / "models").resolve()
os.environ["OCR_MODELS_DIR"] = str(MODELS_DIR)
os.environ["OCR_MAX_RAM_GB"] = "2.0"  # presupuesto de RAM para los modelos
os.environ["OCR_LIMIT"] = "0"  # 0 = los 20 documentos; baja a 5 para iterar rápido
os.environ["OCR_INPUT"] = "image"  # "image" (PNG) o "pdf"
os.environ["OCR_DOWNLOAD"] = "1"  # descarga modelos/artefactos que falten

for k in ("OCR_MODELS_DIR", "OCR_MAX_RAM_GB", "OCR_LIMIT", "OCR_INPUT"):
    print(f"{k:16s}= {os.environ[k]}")

In [ ]:
# 2) Importar ocr.py (añadimos su carpeta al sys.path para un import robusto)
import sys
from pathlib import Path

OCR_DIR = next(
    p
    for p in (Path.cwd(), Path.cwd() / "scripts" / "ocr", Path.cwd().parent)
    if (p / "ocr.py").exists()
)
if str(OCR_DIR) not in sys.path:
    sys.path.insert(0, str(OCR_DIR))

import ocr

print(f"ocr.py cargado desde: {OCR_DIR}")
print(f"Modelos en el catálogo ({len(ocr.MODELS)}):")
for m in ocr.MODELS:
    tag = " (baseline)" if m.is_baseline else ""
    print(f"  · {m.name:<24} [{m.backend}]{tag}")

In [ ]:
# 2b) Información del computador (reusa el helper del benchmark de LLMs)
sysinfo = ocr.print_system_info()

## Correr el benchmark → DataFrame

`ocr.run_benchmark()` carga cada motor (CPU, ≤ 2 GB), procesa los 20 documentos,
escribe `predictions/<modelo>/{n}.txt` por documento y devuelve un `DataFrame`
con una fila por modelo. Es pesado: la primera vez descarga los pesos de cada
motor. **Requiere** que `assets/images/` y `assets/references/` ya existan (ver
`prepare_dataset.py`).

In [ ]:
df = ocr.run_benchmark()
df

In [ ]:
# 4) Librerías de análisis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## Preparación

Descartamos los modelos que no cargaron (omitidos por presupuesto de RAM, librería
ausente o error) y nos quedamos con las filas evaluadas.

In [ ]:
ok = df[(df["n_docs"] > 0) & (df["load_error"] == "")].copy()
ok = ok.sort_values("cer").reset_index(drop=True)
print(f"{len(ok)} modelos evaluados de {len(df)}")
skipped = df[df["load_error"] != ""]
if len(skipped):
    print("\nOmitidos:")
    print(skipped[["model", "load_error"]].to_string(index=False))
ok[
    [
        "model",
        "family",
        "cer",
        "wer",
        "word_f1",
        "term_wer",
        "field_recall",
        "mean_latency_ms",
        "pages_per_sec",
        "ram_peak_mb",
    ]
]

## Selección del mejor modelo

Score compuesto normalizado `[0, 1]` (mayor = mejor):

- **CER** (menor mejor) — calidad de carácter, métrica principal de OCR
- **term-WER** (menor mejor) — calidad en terminología clínica (dosis, fármacos)
- **word-F1** (mayor mejor) — equilibrio precisión/recall a nivel de palabra
- **latencia** (menor mejor) — velocidad por página en CPU
- **RAM peak** (menor mejor) — huella de memoria

Ajusta los pesos según la prioridad del despliegue.

In [ ]:
W = {"cer": 0.30, "term_wer": 0.20, "word_f1": 0.15, "latency": 0.20, "ram": 0.15}


def norm_low(s):  # menor = mejor -> [0,1] con 1 = mejor
    rng = s.max() - s.min()
    return (s.max() - s) / rng if rng else pd.Series(1.0, index=s.index)


def norm_high(s):  # mayor = mejor -> [0,1] con 1 = mejor
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng else pd.Series(1.0, index=s.index)


ok["score"] = (
    W["cer"] * norm_low(ok["cer"])
    + W["term_wer"] * norm_low(ok["term_wer"])
    + W["word_f1"] * norm_high(ok["word_f1"])
    + W["latency"] * norm_low(ok["mean_latency_ms"])
    + W["ram"] * norm_low(ok["ram_peak_mb"])
)
ranking = ok.sort_values("score", ascending=False).reset_index(drop=True)
best = ranking.iloc[0]

print(f"🏆 Mejor modelo: {best['model']}  (score = {best['score']:.3f})")
print(
    f"   CER {best['cer']:.3f} · WER {best['wer']:.3f} · word-F1 {best['word_f1']:.3f}"
)
print(
    f"   term-WER {best['term_wer']:.3f} · field recall {best['field_recall']:.3f} · "
    f"latencia {best['mean_latency_ms']:.0f} ms · RAM peak {best['ram_peak_mb']:.0f} MB"
)
ranking[
    [
        "model",
        "score",
        "cer",
        "wer",
        "word_f1",
        "term_wer",
        "field_recall",
        "mean_latency_ms",
        "ram_peak_mb",
    ]
]

## Imágenes comparativas

Se guardan en `./figures/`.

In [ ]:
FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(exist_ok=True)


def save(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, bbox_inches="tight")
    print("guardado:", path)


BASELINE = df[df["is_baseline"]]["model"].iloc[0] if df["is_baseline"].any() else None


def colors_for(rows):
    return ["#e76f51" if m == BASELINE else "#2a9d8f" for m in rows["model"]]

In [ ]:
# CER vs WER (barras agrupadas) — calidad global
r = ranking.sort_values("cer", ascending=False)
y = np.arange(len(r))
h = 0.4
fig, ax = plt.subplots()
ax.barh(y + h / 2, r["cer"], h, label="CER", color="#264653")
ax.barh(y - h / 2, r["wer"], h, label="WER", color="#e9c46a")
ax.set_yticks(y)
ax.set_yticklabels(r["model"])
ax.set_xlabel("tasa de error (menor es mejor)")
ax.set_title("Calidad global: CER vs WER")
ax.legend()
save(fig, "01_cer_wer.png")
plt.show()

In [ ]:
# term-WER: error sobre terminología clínica (fármacos, dosis, abreviaturas)
r = ranking.sort_values("term_wer", ascending=False)
fig, ax = plt.subplots()
ax.barh(r["model"], r["term_wer"], color=colors_for(r))
ax.set_xlabel("term-WER — error en terminología clínica (menor es mejor)")
ax.set_title("Error sobre entidades clínicas (medicamentos, valores, abreviaturas)")
for i, v in enumerate(r["term_wer"]):
    ax.text(v, i, f" {v:.3f}", va="center")
save(fig, "02_term_wer.png")
plt.show()

In [ ]:
# Precisión / Recall / F1 a nivel de palabra
r = ranking.sort_values("word_f1")
y = np.arange(len(r))
h = 0.26
fig, ax = plt.subplots()
ax.barh(y + h, r["word_precision"], h, label="precision", color="#8ecae6")
ax.barh(y, r["word_recall"], h, label="recall", color="#219ebc")
ax.barh(y - h, r["word_f1"], h, label="F1", color="#023047")
ax.set_yticks(y)
ax.set_yticklabels(r["model"])
ax.set_xlim(0, 1)
ax.set_xlabel("palabra (mayor es mejor)")
ax.set_title("Precisión, recall y F1 a nivel de palabra")
ax.legend()
save(fig, "03_prf1.png")
plt.show()

In [ ]:
# field recall: entidades clínicas de la referencia recuperadas
r = ranking.sort_values("field_recall")
fig, ax = plt.subplots()
ax.barh(r["model"], r["field_recall"], color="#2a9d8f")
ax.set_xlim(0, 1)
ax.set_xlabel("field recall (mayor es mejor)")
ax.set_title("Recuperación de terminología clínica")
for i, v in enumerate(r["field_recall"]):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center")
save(fig, "04_field_recall.png")
plt.show()

In [ ]:
# Latencia: media por página y percentil 95 (rendimiento)
r = ranking.sort_values("mean_latency_ms", ascending=False)
y = np.arange(len(r))
h = 0.4
fig, ax = plt.subplots()
ax.barh(y + h / 2, r["mean_latency_ms"], h, label="latencia media", color="#e76f51")
ax.barh(y - h / 2, r["p95_latency_ms"], h, label="latencia p95", color="#f4a261")
ax.set_yticks(y)
ax.set_yticklabels(r["model"])
ax.set_xlabel("milisegundos por página (menor es mejor)")
ax.set_title("Rendimiento: latencia media vs p95")
ax.legend()
save(fig, "05_latency.png")
plt.show()

In [ ]:
# Throughput: páginas por segundo en CPU
r = ranking.sort_values("pages_per_sec")
fig, ax = plt.subplots()
ax.barh(r["model"], r["pages_per_sec"], color=colors_for(r))
ax.set_xlabel("páginas por segundo (mayor es mejor)")
ax.set_title("Velocidad de inferencia en CPU")
for i, v in enumerate(r["pages_per_sec"]):
    ax.text(v, i, f" {v:.2f}", va="center")
save(fig, "06_throughput.png")
plt.show()

In [ ]:
# RAM peak vs presupuesto de 2 GB
r = ranking.sort_values("ram_peak_mb")
fig, ax = plt.subplots()
ax.barh(r["model"], r["ram_peak_mb"], color=colors_for(r))
ax.axvline(
    ocr.MAX_RAM_GB * 1024,
    color="red",
    ls="--",
    label=f"presupuesto {ocr.MAX_RAM_GB:.0f} GB",
)
ax.set_xlabel("RAM peak (MB) — menor es mejor")
ax.set_title("Pico de memoria durante el OCR")
for i, v in enumerate(r["ram_peak_mb"]):
    ax.text(v, i, f" {v:.0f}", va="center")
ax.legend()
save(fig, "07_ram_peak.png")
plt.show()

In [ ]:
# Tiempo de carga del modelo
r = ranking.sort_values("load_time_ms")
fig, ax = plt.subplots()
ax.barh(r["model"], r["load_time_ms"] / 1000.0, color=colors_for(r))
ax.set_xlabel("tiempo de carga (s) — menor es mejor")
ax.set_title("Tiempo de carga del modelo")
for i, v in enumerate(r["load_time_ms"] / 1000.0):
    ax.text(v, i, f" {v:.1f}", va="center")
save(fig, "08_load_time.png")
plt.show()

In [ ]:
# Trade-off calidad vs velocidad (burbuja = RAM peak, color = score)
fig, ax = plt.subplots()
sizes = (ranking["ram_peak_mb"] / ranking["ram_peak_mb"].max()) * 900 + 80
sc = ax.scatter(
    ranking["mean_latency_ms"],
    ranking["cer"],
    s=sizes,
    c=ranking["score"],
    cmap="viridis",
    alpha=0.85,
    edgecolors="k",
)
for _, row in ranking.iterrows():
    ax.annotate(
        row["model"],
        (row["mean_latency_ms"], row["cer"]),
        fontsize=8,
        xytext=(6, 4),
        textcoords="offset points",
    )
ax.set_xlabel("latencia media (ms/página) — menor mejor")
ax.set_ylabel("CER — menor mejor")
ax.set_title("Calidad vs velocidad (burbuja = RAM peak)")
plt.colorbar(sc, label="score compuesto")
save(fig, "09_tradeoff.png")
plt.show()

In [ ]:
# Heatmap de métricas normalizadas (1 = mejor) por modelo
metrics = [
    "cer",
    "wer",
    "term_wer",
    "word_f1",
    "field_recall",
    "mean_latency_ms",
    "ram_peak_mb",
]
higher_better = {"word_f1", "field_recall"}
norm = pd.DataFrame(index=ranking["model"])
for m in metrics:
    s = ranking[m].values
    rng = s.max() - s.min()
    if m in higher_better:
        norm[m] = (s - s.min()) / rng if rng else 1.0
    else:
        norm[m] = (s.max() - s) / rng if rng else 1.0
fig, ax = plt.subplots(figsize=(max(8, len(metrics)), 0.6 * len(norm) + 2))
im = ax.imshow(norm.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(metrics)))
ax.set_xticklabels(metrics, rotation=45, ha="right")
ax.set_yticks(range(len(norm.index)))
ax.set_yticklabels(norm.index)
for i in range(norm.shape[0]):
    for j in range(norm.shape[1]):
        ax.text(j, i, f"{norm.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="normalizado (1 = mejor)")
ax.set_title("Métricas normalizadas por modelo")
save(fig, "10_heatmap.png")
plt.show()

In [ ]:
# Ranking final por score compuesto — el mejor resaltado
fig, ax = plt.subplots()
cols = ["#2a9d8f" if i == 0 else "#90be6d" for i in range(len(ranking))]
ax.barh(ranking["model"][::-1], ranking["score"][::-1], color=cols[::-1])
ax.set_xlabel("Score compuesto (mayor = mejor)")
ax.set_title(f"Ranking final  —  🏆 {best['model']}")
for i, v in enumerate(ranking["score"][::-1]):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center")
save(fig, "11_ranking.png")
plt.show()

## Selección final del modelo

Resumen ejecutivo con el modelo recomendado y su comparación contra el baseline
del proyecto (Tesseract). Esta es la decisión que se traslada a la memoria del TFM
y al agente OCR (`packages/agent-ocr`).

In [ ]:
print("=" * 60)
print("RESUMEN DEL BENCHMARK OCR")
print("=" * 60)
print(f"Documentos evaluados : {int(best['n_docs'])} (clínicos sintéticos, ReportLab)")
print("Ground-truth         : verbatim exacto del documento (sin sesgo)")
print(f"Modelos evaluados    : {len(ok)} de {len(df)}")
print(
    f"Presupuesto RAM      : {ocr.MAX_RAM_GB} GB · entrada: {os.environ['OCR_INPUT']}"
)
print("-" * 60)
print(f"RECOMENDADO          : {best['model']}")
print(f"  score compuesto    : {best['score']:.3f}")
print(f"  CER / WER          : {best['cer']:.3f} / {best['wer']:.3f}")
print(
    f"  word P / R / F1    : {best['word_precision']:.3f} / {best['word_recall']:.3f} / {best['word_f1']:.3f}"
)
print(f"  term-WER           : {best['term_wer']:.3f}")
print(f"  field recall       : {best['field_recall']:.3f}")
print(f"  latencia media     : {best['mean_latency_ms']:.0f} ms/página")
print(f"  throughput         : {best['pages_per_sec']:.2f} páginas/s")
print(f"  RAM peak           : {best['ram_peak_mb']:.0f} MB")
if BASELINE and (df["model"] == BASELINE).any():
    b = df[df["model"] == BASELINE].iloc[0]
    print("-" * 60)
    print(f"Comparación vs baseline ({BASELINE}):")
    print(f"  CER       {b['cer']:.3f} -> {best['cer']:.3f}")
    print(f"  term-WER  {b['term_wer']:.3f} -> {best['term_wer']:.3f}")
    print(f"  latencia  {b['mean_latency_ms']:.0f} -> {best['mean_latency_ms']:.0f} ms")
print("-" * 60)
print(f"Figuras en : {FIG_DIR}")
print(f"Predicciones en : {ocr.OUT_DIR}")